### Installation

In [1]:
%%capture
# Skip restarting message in Colab
import sys

modules = list(sys.modules.keys())
for x in modules:
    sys.modules.pop(x) if "PIL" in x or "google" in x else None

!pip install unsloth vllm
!pip install --upgrade pillow
# If you are running this notebook on local, you need to install `diffusers` too
# !pip install diffusers
# Temporarily install a specific TRL nightly version
!pip install git+https://github.com/huggingface/trl.git@e95f9fb74a3c3647b86f251b7e230ec51c64b72b

### Unsloth

Use `PatchFastRL` before all functions to patch GRPO and other RL algorithms!

In [ ]:
from unsloth import FastLanguageModel, PatchFastRL

PatchFastRL("GRPO", FastLanguageModel)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


Load up `Llama 3.1 8B Instruct`, and set parameters

In [ ]:
from unsloth import is_bfloat16_supported
# import torch

# print(torch.__version__,unsloth.__version__)
max_seq_length = 512  # Can increase for longer reasoning traces
lora_rank = 16  # Larger rank = smarter, but slower

### using ghe model 3.2-1B

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="meta-llama/Llama-3.2-1B-Instruct",
    max_seq_length=max_seq_length,
    load_in_4bit=True,  # False for LoRA 16bit
    fast_inference=True,  # Enable vLLM fast inference
    max_lora_rank=lora_rank,
    gpu_memory_utilization=0.6,  # Reduce if out of memory
)

model = FastLanguageModel.get_peft_model(
    model,
    r=lora_rank,  # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],  # Remove QKVO if out of memory
    lora_alpha=lora_rank,
    use_gradient_checkpointing="unsloth",  # Enable long context finetuning
    random_state=3407,
)

INFO 02-28 00:50:11 __init__.py:207] Automatically detected platform cuda.
==((====))==  Unsloth 2025.2.15: Fast Llama patching. Transformers: 4.48.3.
   \\   /|    GPU: Tesla T4. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.5.1+cu124. CUDA: 7.5. CUDA Toolkit: 12.4. Triton: 3.1.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.28.post3. FA2 = False]
 "-____-"     Free Apache license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: vLLM loading unsloth/llama-3.2-1b-instruct-unsloth-bnb-4bit with actual GPU utilization = 59.59%
Unsloth: Your GPU has CUDA compute capability 7.5 with VRAM = 14.74 GB.
Unsloth: Using conservativeness = 1.0. Chunked prefill tokens = 512. Num Sequences = 192.
Unsloth: vLLM's KV Cache can use up to 7.71 GB. Also swap space = 2 GB.
WARNING 02-28 00:50:14 config.py:2448] Casting torch.bfloat16 to torch.float16.
INFO 02-28 00:50:38 config.py:549] This model supp

Loading safetensors checkpoint shards:   0% Completed | 0/1 [00:00<?, ?it/s]


Loading safetensors checkpoint shards:   0% Completed | 0/1 [00:00<?, ?it/s]


INFO 02-28 00:50:45 model_runner.py:1115] Loading model weights took 1.0453 GB
INFO 02-28 00:50:45 punica_selector.py:18] Using PunicaWrapperGPU.
INFO 02-28 00:50:49 worker.py:267] Memory profiling takes 2.75 seconds
INFO 02-28 00:50:49 worker.py:267] the current vLLM instance can use total_gpu_memory (14.74GiB) x gpu_memory_utilization (0.60) = 8.78GiB
INFO 02-28 00:50:49 worker.py:267] model weights take 1.05GiB; non_torch_memory takes 0.05GiB; PyTorch activation peak memory takes 0.89GiB; the rest of the memory reserved for KV Cache is 6.80GiB.
INFO 02-28 00:50:49 executor_base.py:111] # cuda blocks: 13932, # CPU blocks: 4096
INFO 02-28 00:50:49 executor_base.py:116] Maximum concurrency for 512 tokens per request: 435.38x
INFO 02-28 00:50:50 model_runner.py:1434] Capturing cudagraphs for decoding. This may lead to unexpected consequences if the model is not static. To run the model in eager mode, set 'enforce_eager=True' or use '--enforce-eager' in the CLI. If out-of-memory error oc

Capturing CUDA graph shapes: 100%|██████████| 27/27 [00:32<00:00,  1.22s/it]

INFO 02-28 00:51:23 model_runner.py:1562] Graph capturing finished in 33 secs, took 0.35 GiB
INFO 02-28 00:51:23 llm_engine.py:436] init engine (profile, create kv cache, warmup model) took 38.08 seconds



Unsloth 2025.2.15 patched 16 layers with 16 QKV layers, 16 O layers and 16 MLP layers.


### Data Prep
<a name="Data"></a>

Using the wikipedia dataset

In [4]:
# import re
# from datasets import load_dataset, Dataset


from datasets import load_dataset

# Load Wikipedia English dataset
wikipedia_dataset = load_dataset("wikipedia", "20220301.simple")

In [34]:
# def preprocess_wiki(example):
#     tokenized = tokenizer(
#         example["text"],
#         truncation=True,
#         padding="max_length",
#         max_length=512
#     )
#     return {
#         "prompt": example["text"],  # Treat the article as a "prompt"
#         "input_ids": tokenized["input_ids"],
#         "attention_mask": tokenized["attention_mask"],
#     }

# # Apply the fix to the dataset
# tokenized_wiki = wikipedia_dataset.map(
#     preprocess_wiki,
#     batched=True,
#     remove_columns=["text"],
#     num_proc=2
# )


# Format into prompts
def format_wiki_as_prompt(example):
    example["prompt"] = f"Tell me about this topic:\n\n{example['text']}"
    return example


formatted_wiki = wikipedia_dataset.map(format_wiki_as_prompt, remove_columns=["text"])


def tokenize_function(examples):
    tokenized_batch = tokenizer(
        examples["prompt"],
        padding="max_length",
        truncation=True,
        max_length=512,
    )

    # Ensure correct processing for batched input
    truncated_prompts = [
        tokenizer.decode(input_ids[:512], skip_special_tokens=True)
        for input_ids in tokenized_batch["input_ids"]
    ]

    return {
        "prompt": truncated_prompts,  # Store trimmed prompts
        "input_ids": [
            ids[:512] for ids in tokenized_batch["input_ids"]
        ],  # Trim input_ids
        "attention_mask": [
            mask[:512] for mask in tokenized_batch["attention_mask"]
        ],  # Trim attention mask
    }


# Apply tokenization with batch processing
tokenized_wiki = formatted_wiki.map(tokenize_function, batched=True)

Map:   0%|          | 0/205328 [00:00<?, ? examples/s]

Jeopardy data

In [50]:
jeopardy_dataset = load_dataset("csv", data_files="/content/JEOPARDY_CSV.csv")


def preprocess_jeopardy(batch):
    """Processes Jeopardy data into structured Q&A format WITHOUT category."""
    questions = batch[" Question"]  # Extract list of questions
    answers = batch[" Answer"]  # Extract list of answers

    formatted_prompts = [f"Question: {q}" for q in questions]  # Format all prompts

    tokenized = tokenizer(
        formatted_prompts, truncation=True, padding="max_length", max_length=512
    )

    return {
        "prompt": formatted_prompts,  # Store human-readable text (as a list!)
        "answer": answers,  # List of answers
        "input_ids": tokenized["input_ids"],  # List of tokenized input IDs
        "attention_mask": tokenized["attention_mask"],  # List of attention masks
    }


# Apply preprocessing
jeopardy_processed = jeopardy_dataset.map(preprocess_jeopardy, batched=True)

Map:   0%|          | 0/216930 [00:00<?, ? examples/s]

riddles

In [51]:
riddle_dataset = load_dataset("csv", data_files="/content/Riddles.csv")

{
    "prompt": "Riddle: I speak without a mouth and hear without ears. I have no body, but I come alive with the wind. What am I?",
    "answer": "An echo",
    "input_ids": [...],  # Tokenized prompt
    "attention_mask": [...],
}


def preprocess_riddles(batch):
    """Processes riddle data into structured Q&A format WITHOUT hint."""
    riddles = batch["Riddle"]  # Extract list of riddles
    answers = batch["Answer"]  # Extract list of answers

    formatted_prompts = [
        f"Riddle: {r}" for r in riddles
    ]  # Format prompts without the hint

    tokenized = tokenizer(
        formatted_prompts, truncation=True, padding="max_length", max_length=512
    )

    return {
        "prompt": formatted_prompts,  # List of prompts
        "answer": answers,  # List of answers
        "input_ids": tokenized["input_ids"],  # List of tokenized input IDs
        "attention_mask": tokenized["attention_mask"],  # List of attention masks
    }


# Apply preprocessing
riddle_processed = riddle_dataset.map(preprocess_riddles, batched=True)

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

join data

In [61]:
# print(riddle_processed['train'])
# print(jeopardy_processed['train'])
# print(tokenized_wiki['train'])

Merging the datasets

In [62]:
# Keep only the necessary columns (no need to add "answer" to Wikipedia)
wikipedia_processed = tokenized_wiki.select_columns(
    ["prompt", "input_ids", "attention_mask"]
)
jeopardy_processed = jeopardy_processed.select_columns(
    ["prompt", "answer", "input_ids", "attention_mask"]
)
riddle_processed = riddle_processed.select_columns(
    ["prompt", "answer", "input_ids", "attention_mask"]
)

In [63]:
from datasets import concatenate_datasets

final_dataset = concatenate_datasets(
    [
        wikipedia_processed["train"],
        jeopardy_processed["train"],
        riddle_processed["train"],
    ],
    axis=0,
)  # Automatically handles missing columns

Reward functions

In [ ]:
# def perplexity_reward_func(prompts, completions, **kwargs):
#     """
#     Reward function based on perplexity. Lower perplexity = higher reward.
#     """
#     rewards = []
#     for completion in completions:
#         # Skip empty completions
#         if not completion or (isinstance(completion, list) and not completion[0]):
#            rewards.append(0.0)
#            continue

#         text = completion if isinstance(completion, str) else completion[0]
#         tokens = tokenizer(text, return_tensors="pt", truncation=True, padding="max_length", max_length=512)

#         tokens = {k: v.to(model.device) for k, v in tokens.items()}  # Move to GPU if needed

#         with torch.no_grad():
#             outputs = model(**tokens, labels=tokens["input_ids"].clone())
#             if hasattr(outputs, "loss") and outputs.loss is not None:
#                 loss = outputs.loss.item()
#                 rewards.append(-loss)  # Lower loss = higher reward
#             else:
#                 rewards.append(0.0)  # Default reward if loss is missing

#     return rewards


def diversity_reward(prompts, completions, **kwargs):
    rewards = []
    for completion in completions:
        words = completion.split()
        unique_words = set(words)
        diversity_score = len(unique_words) / max(
            1, len(words)
        )  # Ratio of unique words
        rewards.append(diversity_score)  # Higher is better
    return rewards


def length_reward(prompts, completions, **kwargs):
    rewards = []
    for completion in completions:
        length = len(completion.split())  # Count words
        if length < 5:  # Avoid one-word responses
            rewards.append(-1)  # Penalize too short
        elif length > 50:  # Penalize too long
            rewards.append(-(length - 50) / 10)
        else:
            rewards.append(1)  # Ideal length
    return rewards

<a name="Train"></a>
### Train the model

Now set up GRPO Trainer and all configurations!

In [27]:
from trl import GRPOConfig, GRPOTrainer

training_args = GRPOConfig(
    use_vllm=True,  # use vLLM for fast inference!
    learning_rate=5e-6,
    adam_beta1=0.9,
    adam_beta2=0.99,
    weight_decay=0.1,
    warmup_ratio=0.1,
    lr_scheduler_type="cosine",
    optim="paged_adamw_8bit",
    logging_steps=1,
    bf16=is_bfloat16_supported(),
    fp16=not is_bfloat16_supported(),
    per_device_train_batch_size=1,
    gradient_accumulation_steps=1,  # Increase to 4 for smoother training
    num_generations=6,  # Decrease if out of memory
    max_prompt_length=256,
    max_completion_length=200,
    # num_train_epochs = 1, # Set to 1 for a full training run
    max_steps=250,
    save_steps=250,
    max_grad_norm=0.1,
    report_to="none",  # Can use Weights & Biases
    output_dir="outputs",
)

And let's run the trainer! If you scroll up, you'll see a table of rewards. The goal is to see the `reward` column increase!

You might have to wait 150 to 200 steps for any action. You'll probably get 0 reward for the first 100 steps. Please be patient!

| Step | Training Loss | reward    | reward_std | completion_length | kl       |
|------|---------------|-----------|------------|-------------------|----------|
| 1    | 0.000000      | 0.125000  | 0.000000   | 200.000000        | 0.000000 |
| 2    | 0.000000      | 0.072375  | 0.248112   | 200.000000        | 0.000000 |
| 3    | 0.000000      | -0.079000 | 0.163776   | 182.500000        | 0.000005 |


In [67]:
# training_args.max_steps = 1000  # Increase steps to prevent early stopping
# training_args.num_train_epochs = 3  # Run longer for better training

training_args.per_device_train_batch_size = 4  # Increase from 1 to 4
training_args.gradient_accumulation_steps = 4  # Helps with small batch sizes

trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    reward_funcs=[
        # perplexity_reward_func,
        # length_reward_func,
        length_reward,  # Keep responses concise but informative
        diversity_reward,  # Prevent excessive repetition
    ],
    args=training_args,
    train_dataset=final_dataset,  # combined dataset
)

trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs = 1
   \\   /|    Num examples = 423,258 | Num Epochs = 1
O^O/ \_/ \    Batch size per device = 4 | Gradient Accumulation steps = 4
\        /    Total batch size = 16 | Total steps = 250
 "-____-"     Number of trainable parameters = 11,272,192


Step,Training Loss,reward,reward_std,completion_length,kl,rewards / length_reward,rewards / diversity_reward
1,0.000100,-4.996260,3.248087,166.697922,0.001485,-5.704167,0.707907
2,0.000000,-4.778123,3.860205,150.052090,0.001034,-5.496875,0.718752
3,0.000000,-4.660887,3.736717,152.604172,0.001222,-5.395833,0.734947
4,0.000100,-5.746666,3.425878,168.041672,0.001257,-6.469792,0.723126
5,0.000100,-4.818887,3.010748,172.281258,0.001347,-5.566667,0.747780
6,0.000100,-5.213428,4.017190,153.239590,0.001786,-5.908334,0.694905
7,0.000000,-4.142578,3.805052,147.937500,0.001190,-4.840625,0.698047
8,0.000100,-4.908747,3.758694,149.614586,0.001604,-5.615625,0.706878
9,0.000100,-5.461813,4.313339,154.604172,0.002464,-6.178125,0.716312
10,0.000100,-4.669522,3.133572,160.916668,0.002937,-5.400000,0.730478


WARNING 02-28 05:57:06 scheduler.py:1091] Input prompt (513 tokens) is too long and exceeds limit of 512
WARNING 02-28 05:57:06 scheduler.py:1091] Input prompt (513 tokens) is too long and exceeds limit of 512
WARNING 02-28 05:57:06 scheduler.py:1091] Input prompt (513 tokens) is too long and exceeds limit of 512
WARNING 02-28 05:57:06 scheduler.py:1091] Input prompt (513 tokens) is too long and exceeds limit of 512
WARNING 02-28 05:57:06 scheduler.py:1091] Input prompt (513 tokens) is too long and exceeds limit of 512
WARNING 02-28 05:57:06 scheduler.py:1091] Input prompt (513 tokens) is too long and exceeds limit of 512


TrainOutput(global_step=250, training_loss=0.018020449372794248, metrics={'train_runtime': 12473.4046, 'train_samples_per_second': 0.321, 'train_steps_per_second': 0.02, 'total_flos': 0.0, 'train_loss': 0.018020449372794248})

<a name="Inference"></a>
### Inference
Now let's try the model we just trained! First, let's first try the model without any GRPO trained:

In [ ]:
from vllm import SamplingParams


text = tokenizer.apply_chat_template(
    [
        {
            "role": "user",
            "content": "What is the riddle of the sphinx, and what are all possible answers satisfying all conditions?",
        },
    ],
    tokenize=False,
    add_generation_prompt=True,
)


sampling_params = SamplingParams(
    temperature=0.8,
    top_p=0.95,
    max_tokens=1024,
)
output = (
    model.fast_generate(
        [text],
        sampling_params=sampling_params,
        lora_request=None,
    )[0]
    .outputs[0]
    .text
)

output

Processed prompts: 100%|██████████| 1/1 [00:07<00:00,  7.09s/it, est. speed input: 7.92 toks/s, output: 60.96 toks/s]


'The Riddle of the Sphinx is an ancient riddle from ancient Greek mythology, said to have been posed by King Rhadamanthus of Athens to the monster Typhon. The riddle was famously inscribed on a stone in the palace of Rhadamanthus, and it is still widely known today as the Sphinx Riddle.\n\nThe riddle is as follows:\n\n"I am wet, I am dry, I have a head and a tail,\nBut no handle, nor mouth, nor any opening wide,\nWhat am I?"\n\nThe answer to this riddle is "a coin," in the sense that a coin has a head on one side and a tail on the other, but it does not have a handle or a mouth to open it.\n\nAs for possible answers satisfying all conditions, here are some examples:\n\n1. Coin: A coin is a perfect example of the riddle. It meets the criteria of having a head on one side and a tail on the other, but it does not have a handle or mouth to open.\n2. Dice: Dice are another example. They have a head on one side (the top) and a tail on the other, but they do not have any handles or openings.\

And now with the LoRA we just trained with GRPO - we first save the LoRA first!

In [69]:
model.save_lora("grpo_saved_lora")

In [ ]:
from vllm import SamplingParams

text = tokenizer.apply_chat_template(
    [
        {
            "role": "user",
            "content": "If it takes 1 hour for 60 people to play an opera, how many hours will it take 600 people to play the same opera?",
        },
        {
            "role": "user",
            "content": "Is a pound of feathers or a British pound heavier?",
        },
        {
            "role": "user",
            "content": "A boy runs down the stairs in the morning and sees a tree in his living room, and some boxes under the tree. What's going on?",
        },
        {"role": "user", "content": "What happens if you crack your knuckles a lot?"},
        {
            "role": "user",
            "content": "If there is a shark in the pool of my basement, is it safe to go upstairs?",
        },
        {
            "role": "user",
            "content": "How much wood could a woodchuck chuck if there were only 5 pounds of wood in the world?",
        },
        {
            "role": "user",
            "content": "Who is the current President of the United States?",
        },
        {"role": "user", "content": "Was Talos alive?"},
        {"role": "user", "content": "How many Ls are in the word parallel?"},
        {
            "role": "user",
            "content": "What is the riddle of the sphinx, and what are all possible answers satisfying all conditions?",
        },
    ],
    tokenize=False,
    add_generation_prompt=True,
)


sampling_params = SamplingParams(
    temperature=0.8,
    top_p=0.95,
    max_tokens=1024,
)

output = (
    model.fast_generate(
        [text],
        sampling_params=sampling_params,
        lora_request=None,
    )[0]
    .outputs[0]
    .text
)

output

Processed prompts: 100%|██████████| 1/1 [00:03<00:00,  3.70s/it, est. speed input: 67.38 toks/s, output: 71.43 toks/s]


'The Riddle of the Sphinx\n\nA priest, a philosopher, and a hunter walked into a bar and ordered a round of drinks. The priest said, "I will give you a riddle. If I solve it, I get a free drink. But if I don\'t, you must give me one."\n\nThe philosopher said, "I accept your riddle, but if I solve it, I must give you a drink. But if I don\'t, I claim victory."\n\nThe hunter said, "I will take a drink."\n\nThe priest said, "Okay, I will now give you the riddle. Here it is: What can be broken, but never held?"\n\nThe philosopher said, "I don\'t know, but I\'ll take a drink if I can solve it."\n\nThe hunter said, "I\'ll take a drink."\n\nThe priest said, "Fine. You get a drink. Here\'s the riddle: What has a head, a tail, but no body?"\n\nThe philosopher said, "A coin."\n\nThe hunter said, "A coin."\n\nThe priest said, "Fine, you get a drink. Here\'s the riddle: What is always coming but never arrives?"\n\nThe hunter said, "Tomorrow."\n\nThe philosopher said, "Tomorrow\'s coming, but it\'s

In [71]:
# Save the fine-tuned model
trainer.save_model("./wiki_tuned")

# Optionally, save the tokenizer too for later reloading
tokenizer.save_pretrained("./wiki_tuned")

('./wiki_tuned/tokenizer_config.json',
 './wiki_tuned/special_tokens_map.json',
 './wiki_tuned/tokenizer.json')

Now we load the LoRA and test:

In [ ]:
from vllm import SamplingParams

text = tokenizer.apply_chat_template(
    [
        # {"role" : "system", "content" : SYSTEM_PROMPT},
        {"role": "user", "content": "Give me a riddle"},
    ],
    tokenize=False,
    add_generation_prompt=True,
)


sampling_params = SamplingParams(
    temperature=0.8,
    top_p=0.95,
    max_tokens=1024,
)
output = (
    model.fast_generate(
        text,
        sampling_params=sampling_params,
        lora_request=model.load_lora("grpo_saved_lora"),
    )[0]
    .outputs[0]
    .text
)

output

In [88]:
from google.colab import drive

drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [91]:
# save_path =  "/content/drive/MyDrive/Misc College/LLMs/assn6data"

# trainer.save_model(save_path)
# tokenizer.save_pretrained(save_path)


save_path2 = "/content/drive/MyDrive/Misc College/LLMs/assn6data_lora"

model.save_lora(save_path2)  # Saves only LoRA adapters
tokenizer.save_pretrained(save_path2)  # Save tokenizer too

Repo card metadata block was not found. Setting CardData to empty.


('/content/drive/MyDrive/Misc College/LLMs/assn6data_lora/tokenizer_config.json',
 '/content/drive/MyDrive/Misc College/LLMs/assn6data_lora/special_tokens_map.json',
 '/content/drive/MyDrive/Misc College/LLMs/assn6data_lora/tokenizer.json')

How to use the lora data

```
from unsloth import FastLanguageModel

# Load base model again
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="meta-llama/Llama-3.2-1B-Instruct",
    max_seq_length=512,
    load_in_4bit=True,
)

# Load LoRA weights back into the base model
model.load_lora("/content/drive/MyDrive/Misc College/LLMs/assn6data_lora")


```



Our reasoning model is much better - it's not always correct, since we only trained it for an hour or so - it'll be better if we extend the sequence length and train for longer!

<a name="Save"></a>
### Saving to float16 for VLLM

We also support saving to `float16` directly. Select `merged_16bit` for float16 or `merged_4bit` for int4. We also allow `lora` adapters as a fallback. Use `push_to_hub_merged` to upload to your Hugging Face account! You can go to https://huggingface.co/settings/tokens for your personal tokens.